In [1]:
import asyncio
import math
import pandas as pd

from itertools import chain, repeat

In [2]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [3]:
from input_output.Standard_Input_and_Output import standard_input, standard_output
from input_output.Class_InputOutput import InputOutput

io = InputOutput()

wb_name = 'IBIT TEMPLATE.xlsx'
cell = 'A1'

wb_name = io.set_xw_book(wb_name)

In [ ]:
from ib_insync import *
from ibkr.Class_IBKR_IB import IBKR_IB
ibkr = IBKR_IB(port=7496)

async def start_ibkr():
    await ibkr.connect()
    print("IBKR connected:", ibkr.ib.isConnected())

await start_ibkr()

IBKR connected: True


Error 1100, reqId -1: Connectivity between IBKR and Trader Workstation has been lost.
Error handling fields: ['10', '4', 'BRR', 'FUT', '20260731 11:00:00 America/New_York', '0', '', 'CME', 'USD', 'BTCN6', 'BTC', 'BTC', '850790355', '5', '5', 'ACTIVETIM,AD,ADJUST,ALERT,ALGO,ALLOC,AVGCOST,BASKET,BENCHPX,COND,CONDORDER,DAY,DEACT,DEACTDIS,DEACTEOD,GAT,GTC,GTD,GTT,HID,ICE,IOC,LIT,LMT,LTH,MIT,MKT,MKTPROT,MTL,NGCOMB,NONALGO,OCA,PEGBENCH,SCALE,SCALERST,SNAPMID,SNAPMKT,SNAPREL,STP,STPLMT,STPPROT,TRAIL,TRAILLIT,TRAILLMT,TRAILMIT,WHATIF', 'CME', '1', '296180912', 'CME CF Bitcoin Reference Rate', '', '202607', '', '', '', 'US/Central', '20260627:CLOSED;20260628:CLOSED;20260626:1602-20260627:0200;20260627:0400-20260629:1600;20260629:1602-20260630:1600;20260630:1602-20260701:1600;20260701:1602-20260702:1600;20260703:CLOSED;20260704:CLOSED;20260705:CLOSED;20260702:1602-20260703:1600;20260703:1602-20260704:0200;20260704:0400-20260706:1600', '20260627:CLOSED;20260628:CLOSED;20260629:0830-20260629:1515;

In [5]:
async def make_chains_from_symbols(symbols_list, product_type):

    contracts     = []
    details       = []
    option_chains = []

    for symbol in symbols_list:

        if product_type == 'equity':
            contract = Stock(symbol=symbol, exchange='SMART', currency="USD")
            fut_exch = ""
        elif product_type == "future":
            contract = Future(localSymbol=symbol, exchange='CME', currency="USD")
            fut_exch = "CME"

        contract = await ibkr.ib.qualifyContractsAsync(contract) # this may lead to a printed line since its return has nowhere to be mapped
        contracts.append(contract[0])

        print(contract[0])

        detail = await ibkr.ib.reqContractDetailsAsync(contract[0])
        details.append(detail[0])

        print(detail[0])

        option_chain = await ibkr.ib.reqSecDefOptParamsAsync(underlyingSymbol=detail[0].contract.symbol,
                                                             futFopExchange=fut_exch,
                                                             underlyingSecType=detail[0].contract.secType,
                                                             underlyingConId=detail[0].contract.conId
                                                            )
        option_chains.append(option_chain)

        print(option_chain)

        # print('count=', len(stocl_option_chain[0].expirations), stock_option_chain[0].expirations)
        # print('count=', len(stock_option_chain[0].strikes), stock_option_chain[0].strikes)
        # print("2 *", len(stock_option_chain[0].expirations), "*", len(stock_option_chain[0].strikes), "=", 
        #                       len(stock_option_chain[0].expirations) * len(stock_option_chain[0].strikes) * 2)

        # print('\n')

    return option_chains, details, contracts

In [6]:
stock_symbols = [] # ['IBIT']

s_chain, s_detail, s_contracts = await make_chains_from_symbols(stock_symbols, 'equity')


future_symbols = [
                  'BTCN6',
                  'BTCQ6',
                  'BTCU6',
                  'BTCV6',
                  'BTCX6',
                  'BTCZ6'
                  ]


f_chain, f_detail, f_contracts = await make_chains_from_symbols(future_symbols, 'future')

combined_zip = chain(zip(s_chain, s_detail, repeat("equity")), zip(f_chain, f_detail, repeat("future")))

Future(conId=850790355, symbol='BRR', lastTradeDateOrContractMonth='20260731', multiplier='5', exchange='CME', currency='USD', localSymbol='BTCN6', tradingClass='BTC')
ContractDetails(contract=Contract(secType='FUT', conId=850790355, symbol='BRR', lastTradeDateOrContractMonth='20260731', multiplier='5', exchange='CME', currency='USD', localSymbol='BTCN6', tradingClass='BTC'), marketName='BTC', minTick=5.0, orderTypes='ACTIVETIM,AD,ADJUST,ALERT,ALGO,ALLOC,AVGCOST,BASKET,BENCHPX,COND,CONDORDER,DAY,DEACT,DEACTDIS,DEACTEOD,GAT,GTC,GTD,GTT,HID,ICE,IOC,LIT,LMT,LTH,MIT,MKT,MKTPROT,MTL,NGCOMB,NONALGO,OCA,PEGBENCH,SCALE,SCALERST,SNAPMID,SNAPMKT,SNAPREL,STP,STPLMT,STPPROT,TRAIL,TRAILLIT,TRAILLMT,TRAILMIT,WHATIF', validExchanges='CME', priceMagnifier=1, underConId=296180912, longName='CME CF Bitcoin Reference Rate', contractMonth='202607', industry='', category='', subcategory='', timeZoneId='US/Central', tradingHours='20260627:CLOSED;20260628:CLOSED;20260626:1602-20260627:0200;20260627:0400-2026

In [7]:
option_contracts = []

for chain, details, product_type in combined_zip:
    print(chain[0])
    #print(chain.strikes)
    
    for expiry in chain[0].expirations:
        for strike in chain[0].strikes:
            for right in ['C', 'P']:
                
                if product_type == 'equity':
                    option_contract = Option(lastTradeDateOrContractMonth=expiry,
                                                    strike=float(strike),
                                                    right=right,
                                                    symbol=details.contract.symbol,
                                                    exchange=details.contract.exchange,
                                                    currency=details.contract.currency,
                                                    )
                
                elif product_type == 'future':
                    option_contract = FuturesOption(lastTradeDateOrContractMonth=expiry,
                                                    strike=float(strike),
                                                    right=right,
                                                    symbol=details.contract.symbol,
                                                    exchange=details.contract.exchange,
                                                    currency=details.contract.currency,
                                                    )
                option_contracts.append(option_contract)

option_contracts = await ibkr.ib.qualifyContractsAsync(*option_contracts) # this may lead to printed lines since its return has nowhere to be mapped

# print(len(option_contracts))

OptionChain(exchange='CME', underlyingConId='850790355', tradingClass='BTC', multiplier='5', expirations=['20260731'], strikes=[10000.0, 17500.0, 20000.0, 22500.0, 25000.0, 27500.0, 30000.0, 32500.0, 35000.0, 37500.0, 40000.0, 42500.0, 45000.0, 47500.0, 48000.0, 48500.0, 49000.0, 49500.0, 50000.0, 50500.0, 51000.0, 51500.0, 52000.0, 52500.0, 53000.0, 53500.0, 54000.0, 54500.0, 55000.0, 55250.0, 55500.0, 55750.0, 56000.0, 56250.0, 56500.0, 56750.0, 57000.0, 57250.0, 57500.0, 57750.0, 58000.0, 58250.0, 58500.0, 58750.0, 59000.0, 59250.0, 59500.0, 59750.0, 60000.0, 60250.0, 60500.0, 60750.0, 61000.0, 61250.0, 61500.0, 61750.0, 62000.0, 62250.0, 62500.0, 62750.0, 63000.0, 63250.0, 63500.0, 63750.0, 64000.0, 64250.0, 64500.0, 64750.0, 65000.0, 65250.0, 65500.0, 65750.0, 66000.0, 66250.0, 66500.0, 66750.0, 67000.0, 67250.0, 67500.0, 67750.0, 68000.0, 68250.0, 68500.0, 68750.0, 69000.0, 69250.0, 69500.0, 69750.0, 70000.0, 70250.0, 70500.0, 70750.0, 71000.0, 71250.0, 71500.0, 71750.0, 72000.0,

In [8]:
def clean_num(x):
    """
    Convert IBKR nan / -1 / None style missing values to None.
    """
    if x is None:
        return None
    try:
        if math.isnan(x):
            return None
    except TypeError:
        pass
    if x == -1:
        return None
    return x

In [9]:
def ticker_row(ticker):
    contract = ticker.contract
    return {
            "conId": getattr(ticker.contract, "conId", None),
            "secType": getattr(contract, "secType", None),
            "symbol": getattr(contract, "symbol", None),
            "expiration": getattr(contract, "lastTradeDateOrContractMonth", None),
            "right": getattr(contract, "right", None),
            "strike": getattr(contract, "strike", None),

            "close": clean_num(getattr(ticker, "close", None)),
        #    "volume": clean_num(getattr(ticker, "volume", None)),
            "avgVolume": clean_num(getattr(ticker, "avVolume", None)),
            "futuresOpenInterest": clean_num(getattr(ticker, "futuresOpenInterest", None)),
            "putOpenInterest": clean_num(getattr(ticker, "putOpenInterest", None)),
            "callOpenInterest": clean_num(getattr(ticker, "callOpenInterest", None)),
        }

In [10]:
async def get_data(contracts, batch_size=50, wait_seconds=30):
    
    generic_ticks = "100,101,165,588"
    all_rows = []

    for i in range(0, len(contracts), batch_size):
        batch = contracts[i:i + batch_size]
        tickers = []

        print(f"Requesting {i} to {i + len(batch) - 1} of {len(contracts)}")

        for contract in batch:
            ticker = ibkr.ib.reqMktData(
                contract,
                genericTickList=generic_ticks,
                snapshot=False,
            )
            tickers.append(ticker)

        await asyncio.sleep(wait_seconds)

        rows = [ticker_row(ticker) for ticker in tickers]
        all_rows.extend(rows)

        # IMPORTANT: cancel streaming data before next batch
        for ticker in tickers:
            ibkr.ib.cancelMktData(ticker.contract)

        await asyncio.sleep(2)  # small pause between batches

    return pd.DataFrame(all_rows)

In [11]:
linear_contracts = [*f_contracts, *s_contracts] 
df_linear  = await get_data(linear_contracts, batch_size = 25, wait_seconds=60)
df_linear

Requesting 0 to 5 of 6


,conId,secType,symbol,expiration,right,strike,close,avgVolume,futuresOpenInterest,putOpenInterest,callOpenInterest
0,850790355,FUT,BRR,20260731,,0.0,59995.0,None,0.0,0.0,0.0
1,859040542,FUT,BRR,20260828,,0.0,60240.0,None,0.0,0.0,0.0
2,772435574,FUT,BRR,20260925,,0.0,60445.0,None,0.0,0.0,0.0
3,876880607,FUT,BRR,20261030,,0.0,60675.0,None,0.0,0.0,0.0
4,887699043,FUT,BRR,20261127,,0.0,60975.0,None,0.0,0.0,0.0
5,751356958,FUT,BRR,20261224,,0.0,61325.0,None,0.0,0.0,0.0


In [12]:
df_options = await get_data(option_contracts, batch_size = 25, wait_seconds=60)
df_options

Requesting 0 to 24 of 2940
Requesting 25 to 49 of 2940
Requesting 50 to 74 of 2940
Requesting 75 to 99 of 2940
Requesting 100 to 124 of 2940
Requesting 125 to 149 of 2940
Requesting 150 to 174 of 2940
Requesting 175 to 199 of 2940
Requesting 200 to 224 of 2940
Requesting 225 to 249 of 2940
Requesting 250 to 274 of 2940
Requesting 275 to 299 of 2940
Requesting 300 to 324 of 2940
Requesting 325 to 349 of 2940
Requesting 350 to 374 of 2940
Requesting 375 to 399 of 2940
Requesting 400 to 424 of 2940
Requesting 425 to 449 of 2940
Requesting 450 to 474 of 2940
Requesting 475 to 499 of 2940
Requesting 500 to 524 of 2940
Requesting 525 to 549 of 2940
Requesting 550 to 574 of 2940
Requesting 575 to 599 of 2940
Requesting 600 to 624 of 2940
Requesting 625 to 649 of 2940
Requesting 650 to 674 of 2940
Requesting 675 to 699 of 2940
Requesting 700 to 724 of 2940
Requesting 725 to 749 of 2940
Requesting 750 to 774 of 2940
Requesting 775 to 799 of 2940
Requesting 800 to 824 of 2940
Requesting 825 to 8

,conId,secType,symbol,expiration,right,strike,close,avgVolume,futuresOpenInterest,putOpenInterest,callOpenInterest
0,851209519,FOP,BRR,20260731,C,10000.0,49820.0,None,None,0.0,0.0
1,851209584,FOP,BRR,20260731,P,10000.0,0.0,None,None,0.0,0.0
2,852657744,FOP,BRR,20260731,C,17500.0,42350.0,None,None,0.0,0.0
3,852658044,FOP,BRR,20260731,P,17500.0,5.0,None,None,0.0,0.0
4,851535783,FOP,BRR,20260731,C,20000.0,39860.0,None,None,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
2935,756512288,FOP,BRR,20261224,P,500000.0,430500.0,None,None,0.0,0.0
2936,756512281,FOP,BRR,20261224,C,550000.0,0.0,None,None,0.0,0.0
2937,756512251,FOP,BRR,20261224,P,550000.0,479565.0,None,None,0.0,0.0
2938,756512303,FOP,BRR,20261224,C,600000.0,0.0,None,None,0.0,0.0


In [13]:
df_option_conIds = (
    df_options.pivot_table(
        index=["symbol", "strike"],
        columns=["expiration", "right"],
        values="conId",
        aggfunc="first"
    )
    .sort_index()
    .sort_index(axis=1)
    .reset_index()
)

df_option_conIds

expiration symbol    strike     20260731                  20260828  \
right                                  C            P            C   
0             BRR    1000.0          NaN          NaN          NaN   
1             BRR    5000.0          NaN          NaN          NaN   
2             BRR   10000.0  851209519.0  851209584.0  859510025.0   
3             BRR   17500.0  852657744.0  852658044.0  859508335.0   
4             BRR   20000.0  851535783.0  851535798.0  859509215.0   
..            ...       ...          ...          ...          ...   
331           BRR  405000.0          NaN          NaN          NaN   
332           BRR  450000.0          NaN          NaN          NaN   
333           BRR  500000.0          NaN          NaN          NaN   
334           BRR  550000.0          NaN          NaN          NaN   
335           BRR  600000.0          NaN          NaN          NaN   

expiration                  20260925                  20261030               \
right                 P            C            P            C            P   
0                   NaN  772689836.0  772689948.0          NaN          NaN   
1                   NaN  772689735.0  772689670.0          NaN          NaN   
2           859509792.0  772689866.0  772689717.0  877338257.0  877338335.0   
3           859509893.0  852656091.0  852656074.0          NaN          NaN   
4           859509945.0  772689756.0  772689673.0  877338260.0  877338325.0   
..                  ...          ...          ...          ...          ...   
331                 NaN  820593552.0  820593340.0          NaN          NaN   
332                 NaN  799038165.0  799038170.0          NaN          NaN   
333                 NaN  799038148.0  799038158.0          NaN          NaN   
334                 NaN  799038175.0  799038179.0          NaN          NaN   
335                 NaN  799038155.0  799038160.0          NaN          NaN   

expiration     20261127                  20261224               
right                 C            P            C            P  
0                   NaN          NaN  751506475.0  751506420.0  
1                   NaN          NaN  751506410.0  751506461.0  
2           888428735.0  888429682.0  751506435.0  751506465.0  
3                   NaN          NaN  852657228.0  852657834.0  
4                   NaN          NaN  765088022.0  765087976.0  
..                  ...          ...          ...          ...  
331                 NaN          NaN  820593399.0  820593414.0  
332                 NaN          NaN  756512269.0  756512244.0  
333                 NaN          NaN  756512284.0  756512288.0  
334                 NaN          NaN  756512281.0  756512251.0  
335                 NaN          NaN  756512303.0  756512278.0  

[336 rows x 14 columns]

In [ ]:
df_option_prices = (
    df_options.pivot_table(
        index=["symbol", "strike"],
        columns=["expiration", "right"],
        values="close",
        aggfunc="first"
    )
    .sort_index()
    #.sort_index(axis=1)
    .reset_index()
)

df_option_prices

expiration symbol    strike 20260731      20260828       20260925            \
right                              C    P        C     P        C         P   
0             BRR    1000.0      NaN  NaN      NaN   NaN  58900.0       0.0   
1             BRR    5000.0      NaN  NaN      NaN   NaN  54935.0       0.0   
2             BRR   10000.0  49820.0  0.0  49925.0   0.0  49985.0       0.0   
3             BRR   17500.0  42350.0  5.0  42485.0  15.0  42575.0      25.0   
4             BRR   20000.0  39860.0  5.0  40010.0  25.0  40115.0      45.0   
..            ...       ...      ...  ...      ...   ...      ...       ...   
331           BRR  405000.0      NaN  NaN      NaN   NaN      5.0  341390.0   
332           BRR  450000.0      NaN  NaN      NaN   NaN      0.0  385975.0   
333           BRR  500000.0      NaN  NaN      NaN   NaN      0.0  435515.0   
334           BRR  550000.0      NaN  NaN      NaN   NaN      0.0  485055.0   
335           BRR  600000.0      NaN  NaN      NaN   NaN      0.0  534600.0   

expiration 20261030       20261127       20261224            
right             C     P        C     P        C         P  
0               NaN   NaN      NaN   NaN  59200.0       0.0  
1               NaN   NaN      NaN   NaN  55275.0       0.0  
2           50030.0   5.0  50255.0  85.0  50395.0      25.0  
3               NaN   NaN      NaN   NaN  43160.0     150.0  
4           40245.0  90.0      NaN   NaN  40775.0     220.0  
..              ...   ...      ...   ...      ...       ...  
331             NaN   NaN      NaN   NaN      5.0  337275.0  
332             NaN   NaN      NaN   NaN      5.0  381435.0  
333             NaN   NaN      NaN   NaN      5.0  430500.0  
334             NaN   NaN      NaN   NaN      0.0  479565.0  
335             NaN   NaN      NaN   NaN      0.0  528635.0  

[336 rows x 14 columns]

: 

In [ ]:
ws_name = 'linear info'
ws_name, range = io.set_xw_sheet_and_range(wb_name, ws_name, cell)
io.print_xw_df(range, df_linear, headerRows=1)

ws_name = 'option info'
ws_name, range = io.set_xw_sheet_and_range(wb_name, ws_name, cell)
io.print_xw_df(ws_name, range, df_options, headerRows=1)

ws_name = 'option conIds' 
ws_name, range = io.set_xw_sheet_and_range(wb_name, ws_name, cell)
io.print_xw_df(range, df_option_conIds, headerRows=1)

ws_name = 'option prices'
ws_name, range = io.set_xw_sheet_and_range(wb_name, ws_name, cell)
io.print_xw_df(range, df_option_prices, headerRows=1)